# 01 — Exploratory Data Analysis
**Olist Brazilian E-Commerce | 100K+ real orders 2016-2018**

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")


## 1. Dataset overview

In [ ]:

tables = ['fact_orders','dim_customers','dim_products','dim_sellers','dim_date','dim_geolocation']
print(f"{'Table':<25} {'Rows':>10}")
print("-"*38)
for t in tables:
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM olist.{t}")).scalar()
    print(f"  olist.{t:<20} {n:>10,}")


## 2. KPI summary

In [ ]:

sql = '''SELECT COUNT(DISTINCT order_id) AS total_orders,
COUNT(DISTINCT customer_id) AS unique_customers,
ROUND(SUM(revenue)::numeric,2) AS total_revenue,
ROUND(AVG(price)::numeric,2) AS avg_order_value,
ROUND(AVG(review_score)::numeric,2) AS avg_review_score,
ROUND(SUM(CASE WHEN is_delivered=1 THEN 1.0 ELSE 0 END)/COUNT(*)*100,2) AS delivery_rate_pct
FROM olist.fact_orders'''
with engine.connect() as conn:
    kpis = pd.read_sql(text(sql),conn).iloc[0]
for k,v in kpis.items():
    print(f"  {k:<28} {v:>15,}")


## 3. Monthly revenue trend

In [ ]:

sql = "SELECT yyyymm, ROUND(SUM(revenue)::numeric,2) AS revenue, COUNT(DISTINCT order_id) AS orders FROM olist.fact_orders GROUP BY yyyymm ORDER BY yyyymm"
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.bar(df['yyyymm'], df['revenue']/1e6, color='steelblue', alpha=0.7, label='Revenue ($M)')
ax2.plot(df['yyyymm'], df['orders'], color='orange', lw=2, marker='o', ms=3, label='Orders')
ax1.set_ylabel('Revenue ($M)'); ax2.set_ylabel('Orders')
ax1.set_title('Monthly Revenue & Orders 2016-2018')
plt.xticks(df['yyyymm'][::3], rotation=45)
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout(); plt.show()
print(f"Peak month: {df.loc[df['revenue'].idxmax(),'yyyymm']}  ->  ${df['revenue'].max():,.2f}")


## 4. Revenue by state

In [ ]:

sql = '''SELECT c.state, COUNT(DISTINCT f.order_id) AS orders, ROUND(SUM(f.revenue)::numeric,2) AS revenue
FROM olist.fact_orders f JOIN olist.dim_customers c ON f.customer_id=c.customer_id
GROUP BY c.state ORDER BY revenue DESC LIMIT 12'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(14,5))
axes[0].barh(df['state'][::-1], df['revenue'][::-1]/1e6, color='steelblue')
axes[0].set_title('Revenue by State ($M)')
axes[1].barh(df['state'][::-1], df['orders'][::-1], color='darkorange')
axes[1].set_title('Orders by State')
plt.tight_layout(); plt.show()


## 5. Revenue by category

In [ ]:

sql = '''SELECT category, ROUND(SUM(revenue)::numeric,2) AS revenue, ROUND(AVG(review_score)::numeric,2) AS avg_review, COUNT(DISTINCT order_id) AS orders
FROM olist.fact_orders WHERE category IS NOT NULL GROUP BY category ORDER BY revenue DESC LIMIT 15'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(16,6))
axes[0].barh(df['category'][::-1], df['revenue'][::-1]/1e6, color='steelblue')
axes[0].set_title('Top 15 Categories by Revenue')
axes[1].scatter(df['revenue']/1e6, df['avg_review'], s=df['orders']/20, alpha=0.7)
for _,row in df.iterrows():
    axes[1].annotate(row['category'][:12],(row['revenue']/1e6,row['avg_review']),fontsize=7,alpha=0.8)
axes[1].set_xlabel('Revenue ($M)'); axes[1].set_ylabel('Avg Review')
axes[1].set_title('Revenue vs Review Score')
plt.tight_layout(); plt.show()


## 6. Review score distribution

In [ ]:

sql = "SELECT review_score, COUNT(*) AS count FROM olist.fact_orders GROUP BY review_score ORDER BY review_score"
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
bars = plt.bar(df['review_score'], df['count'], color=colors, edgecolor='white')
plt.title('Review Score Distribution'); plt.xlabel('Score'); plt.ylabel('Count')
for bar,cnt in zip(bars,df['count']):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200, f'{cnt:,}', ha='center', fontsize=10)
plt.tight_layout(); plt.show()


## 7. Payment type analysis

In [ ]:

sql = '''SELECT payment_type, COUNT(DISTINCT order_id) AS orders, ROUND(SUM(revenue)::numeric,2) AS revenue, ROUND(AVG(payment_installments)::numeric,1) AS avg_installments
FROM olist.fact_orders WHERE payment_type IS NOT NULL GROUP BY payment_type ORDER BY revenue DESC'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(13,5))
axes[0].pie(df['revenue'], labels=df['payment_type'], autopct='%1.1f%%', colors=sns.color_palette('Blues_d',len(df)))
axes[0].set_title('Revenue by Payment Type')
axes[1].bar(df['payment_type'], df['avg_installments'], color='steelblue')
axes[1].set_title('Avg Installments by Payment Type')
plt.tight_layout(); plt.show()


## 8. Delivery performance

In [ ]:

sql = '''SELECT ROUND(AVG(delivery_delay_days)::numeric,1) AS avg_delay,
SUM(CASE WHEN delivery_delay_days>0 THEN 1 ELSE 0 END) AS late,
SUM(CASE WHEN delivery_delay_days<=0 THEN 1 ELSE 0 END) AS ontime,
COUNT(*) AS total FROM olist.fact_orders WHERE is_delivered=1 AND delivery_delay_days IS NOT NULL'''
with engine.connect() as conn:
    s = pd.read_sql(text(sql),conn).iloc[0]
plt.pie([s['ontime'],s['late']], labels=['On-time','Late'], autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'])
plt.title(f'Delivery Performance (avg delay: {s["avg_delay"]} days)')
plt.tight_layout(); plt.show()
print(f"On-time: {s['ontime']:,} ({s['ontime']/s['total']*100:.1f}%)  Late: {s['late']:,} ({s['late']/s['total']*100:.1f}%)")


## Key insights
- SP generates ~30% of all revenue
- Electronics and Automotive dominate categories
- 5-star reviews are most common
- ~30% of orders are delivered late
- Credit card is the dominant payment method